# Transformers

In [1]:
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Project Description

In this project I will go over transformer language models. Specifically I will be explaining the concepts introduced in the paper [Attention Is All You Need](https://arxiv.org/pdf/1706.03762). This is a very important paper that not only changed the field of computer science but also non technical fields. The notebook will cover the architecture which includes encoder, decoder, attention, etc. I will also be covering the math behind it as well as training. I have also implemented my own transformer using Jax. The code for that can be found in the `./src`and `main.py`. Lets get into it! 

## Background

One of the many advantages of transformer models is that they can process large amounts of text data at once. Previously Long Short Term Memory (LSTM) networks/recurrent networks were used to model language but one of their flaws is that they processed data one word (token) at a time. This made it difficult for models to understand text from the beginning of a sequence if the sequence was large, suffered from vanishing gradients, and processed data one token at a time which is computational expensive for large data sets. With the transformer architecture models are capable of better understanding data throughout the sequence and process large amounts of data at once. The specific task discussed in the paper was language translation. Which we will also try to implement. Usually for tasks like text generation like GPT models, only the decoder part of the model is used. Because I want to understand the transformer model thoroughly I want to implement both the encoder and decoder.

## Model Architecture

The transformer architecture from the paper Attention Is All You Need consists of a N decoder and encoder blocks. The encoder block consits of 

### Input Embeddings
The Encoder takes embeddings as input. Embeddings are vectors representations of a word/token. The embeddings are learnable parameters. In the original paper each token embedding is of size $512$ defined as $d_{model}$. Here is an example of some text and what the tokens might look like.  

- **Sequence:**  lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod
tempor incididunt ut labore et dolore magna aliqua.
- **Tokens:**  (lorem ipsum dolor) (sit) (amet, consectetur) (adipiscing elit, sed do) (eiusmod) (tempor) (incididunt) (ut labore) (et dolore magna aliqua.)


- **Sequence:**  Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo
consequat.
- **Tokens:**  (Ut) (enim ad minim veniam,) (quis nostrud) (exercitation) (ullamco) (laboris) (nisi ut aliquip) (ex ea) (commodo consequat.)


Although tokens can be of any size, its easier to understand the attention mechanism if we only think of tokens as one word so in this notebook each token is just one word. Each token is also mapped to a id so that we so we can later look them up in our vocbulary. This is an example of what a token embedding might look like. 

- **Token:**  Dog
- **Token ID:** [12]
- **Embedding:** $\{e_1, e_2, \dots , e_{512} \}$


All together our input embedding is a matrix of size $(\text{seq len}, d_{model})$. This is pretty straight forward. It just means we have a sequence of tokens of size $ \text{seq len} $ and each is represented by a vector of size $d_{model}$.

I used the sentence piece tokenizer to get tokens. Tokens are also learnable but I will not go into detail about them here. You can read more about the sentence piece tokenzier [here](https://github.com/google/sentencepiece).

### Positional Embeddings

Positional embeddings give information about the positon of the token in the sequence. 

The original paper uses trig for the positional embeddings. Essentially 

### Residual Connections
The original paper used residual connections after each attention and linear layer. In my implementation we did it before because the tutorial I followed for implemenation said it was most commonly done this way. The residual connection in the original paper is defined as $LayerNorm(x + SubLayer(x))$. 

Instead we do 
$$
\begin{aligned}
x_{norm} &= LayerNorm(x) \\[4pt]
SubLayerOutput &= SubLayer(x_{norm}) \\[4pt]
x &= x + DropOut(SubLayerOutput)
\end{aligned}
$$

**LayerNorm**

Here we go over a brief of what LayerNorm does. Lets say we have samples with shape `(batch_size=3, seq_len=5, d_model=10)`. We have 3 seqeunces each of size 5 where each token in the sequence has a embedding size of 10. Then what we will do is calculate the mean and variance for each token embedding and normalize it. Lets go over an example.

In [2]:
batch = np.random.rand(3, 5, 10) # Create an array with 3 5x10 arrays

In [3]:
batch.shape

(3, 5, 10)

In [4]:
batch

array([[[0.54648278, 0.54936022, 0.46805969, 0.07198258, 0.23517513,
         0.77768484, 0.22245891, 0.19825211, 0.25649556, 0.5544154 ],
        [0.53370422, 0.6702937 , 0.41250352, 0.71981051, 0.32606003,
         0.6165208 , 0.84685674, 0.35046926, 0.67876903, 0.46051104],
        [0.3925982 , 0.36945105, 0.25557696, 0.26839392, 0.78444393,
         0.84903217, 0.67296443, 0.8190655 , 0.65115516, 0.38867642],
        [0.167819  , 0.38339445, 0.34507844, 0.56092182, 0.71651956,
         0.72667177, 0.17906804, 0.25826887, 0.82166469, 0.06519832],
        [0.66815799, 0.80455987, 0.68219563, 0.41294655, 0.02077717,
         0.55109828, 0.67202348, 0.19823834, 0.38142079, 0.94424235]],

       [[0.7038917 , 0.78411479, 0.92033724, 0.74059813, 0.90423574,
         0.32194835, 0.94775914, 0.52294771, 0.70740174, 0.13673441],
        [0.16409496, 0.54564472, 0.7690766 , 0.01003314, 0.07179655,
         0.32604452, 0.39332593, 0.40191608, 0.30313815, 0.52639634],
        [0.43503743, 0.90

In [15]:
mean = np.mean(batch, axis=-1, keepdims=True) # calculate the mean along the last dimension
variance = np.var(batch, axis=-1, keepdims=True) # calculate the mean along the last dimension

In [16]:
mean # the mean for each token in the sequence for each example

array([[[0.38803672],
        [0.56154989],
        [0.54513577],
        [0.4224605 ],
        [0.53356605]],

       [[0.6689969 ],
        [0.3511467 ],
        [0.6323945 ],
        [0.58648865],
        [0.447145  ]],

       [[0.50524161],
        [0.5322283 ],
        [0.50145762],
        [0.53352367],
        [0.57617176]]])

In [17]:
mean.shape

(3, 5, 1)

In [18]:
variance # the variance for each token in the sequence for each example

array([[[0.04410358],
        [0.02682433],
        [0.04912819],
        [0.06437422],
        [0.07148706]],

       [[0.06423655],
        [0.04798822],
        [0.0936679 ],
        [0.04515944],
        [0.04483601]],

       [[0.08601411],
        [0.06833899],
        [0.08981858],
        [0.07553525],
        [0.0224379 ]]])

In [19]:
variance.shape

(3, 5, 1)

In [20]:
batch = (batch - mean)/ np.sqrt(variance) # normalize

In [21]:
batch.shape

(3, 5, 10)

In [22]:
batch

array([[[ 0.7544744 ,  0.76817596,  0.38104628, -1.50495865,
         -0.72788279,  1.85539194, -0.78843378, -0.90369959,
         -0.62636104,  0.79224727],
        [-0.17001733,  0.66395723, -0.91003256,  0.96629206,
         -1.43783066,  0.335636  ,  1.74199838, -1.28879522,
          0.71570505, -0.61691295],
        [-0.68819489, -0.79262656, -1.30638567, -1.24856012,
          1.0796727 ,  1.37107169,  0.57671711,  1.23587282,
          0.47832148, -0.70588855],
        [-1.00362894, -0.15397262, -0.30498907,  0.54572329,
          1.15898701,  1.19900034, -0.95929263, -0.64713518,
          1.57339983, -1.40809201],
        [ 0.50339089,  1.01355117,  0.55589343, -0.45113216,
         -1.91789521,  0.06557279,  0.51784829, -1.25416799,
         -0.56904249,  1.5359813 ]],

       [[ 0.13767962,  0.4542048 ,  0.99167894,  0.28250713,
          0.92814948, -1.36930161,  1.09987375, -0.57624613,
          0.15152872, -2.1000747 ],
        [-0.85387526,  0.88786691,  1.90781441, -1

We will be referencing these two things often. 

### Encoder

Lets get into the encoder. The encoder part of the model is made up of "N = 6 identical layers". This means we repeat the encoder 6 or N times. Here is an image of the architecture from the paper. 


![title](images/encoder.png)

As mentioed earlier we have a slightly different a architecture. Our architecure follows this.

$$
\begin{aligned}
x_{\text{norm}} &= \operatorname{LayerNorm}(x), \\[4pt]
\text{MultiHeadAttentionBlockOutput} &= \operatorname{MultiHeadAttentionBlock} 
        \left(
       q=x_{\text{norm}},
       k=x_{\text{norm}},
       v=x_{\text{norm}}
       \right), \\[4pt]
x &= x + \operatorname{Dropout}
     \left(\text{MultiHeadAttentionBlockOutput}\right), \\[8pt]
x_{\text{norm}} &= \operatorname{LayerNorm}(x), \\[4pt]
\text{FeedForwardBlockOutput}
    &= \operatorname{FeedForwardBlock}(x_{\text{norm}}), \\[4pt]
x &= x + \operatorname{Dropout}
     \left(\text{FeedForwardBlockOutput}\right).
\end{aligned}
$$
The final output is then passed into the decoder. The encoder is responsible for

### Decoder
The encoder architecture is very similar. In the paper it is defined as this image below. 

![title](images/decoder.png)

Our implementation is slightly different too. Here is our architecture. 
$$
\begin{aligned}
x_{\text{norm}} &= \operatorname{LayerNorm}(x), \\[4pt]
\text{MaskedMultiHeadAttentionBlockOutput} &= \operatorname{MaskedMultiHeadAttentionBlock} 
        \left(
       q=x_{\text{norm}},
       k=x_{\text{norm}},
       v=x_{\text{norm}}
       \right), \\[4pt]
x &= x + \operatorname{Dropout}
     \left(\text{MaskedMultiHeadAttentionBlockOutput}\right), \\[8pt]
x_{\text{norm}} &= \operatorname{LayerNorm}(x), \\[4pt]
\text{CrossMultiHeadAttentionBlockOutput} &= \operatorname{CrossMultiHeadAttentionBlock} 
        \left(
       q=x_{\text{norm}},
       k={\text{encoder\_output}},
       v={\text{encoder\_output}}
       \right), \\[4pt]
x &= x + \operatorname{Dropout}
     \left(\text{CrossMultiHeadAttentionBlockOutput}\right), \\[8pt]
x_{\text{norm}} &= \operatorname{LayerNorm}(x), \\[4pt]
\text{FeedForwardBlockOutput}
    &= \operatorname{FeedForwardBlock}(x_{\text{norm}}), \\[4pt]
x &= x + \operatorname{Dropout}
     \left(\text{FeedForwardBlockOutput}\right).
\end{aligned}
$$

The encoder is responsible for 

Weve covered residual connections, layer norm, input and positional embeddings. Now lets go over attention. 

#### Attention
Attention is defined as.  
$$ Attention(Q, K, V) = softmax(\frac{QK^{T}}{\sqrt{d_k}})V $$

This equation may look very confusing so lets break it down part by part. 

We have a input embeddings of size (${seq\_len}$, $d_{model}$). For a basic visual example lets say "The dog ate" is our input. Then we would have a matrix of shape (${seq\_len}$, $d_{model}$) but only the first 3 rows are used for the embeddings of our sequence "The dog ate". 

Here are our defined parameters ${seq\_len}=5$, $d_{model}=10$, $h heads = 5$ and $d_k=\frac{d_{model}}{h heads} = \frac{10}{5} = 2$ 

If we remeber back to this diagram we see that we pass the input 3 times. This is Q, K, V. So lets begin

In [2]:
q = np.matrix([
    [8 , 63 , 39 , 36 , 81 , 3 , 73 , 8 , 50 , 2],
    [61 , 90 , 99 , 92 , 42 , 25 ,  30 , 5 , 4, 79],
    [11 ,  99 , 10 , 34 , 72 , 9 , 74 , 84 , 65 , 62],
    [0 , 0 , 0 , 0 , 0 , 0 , 0 , 0 , 0 , 0],
    [0 , 0 , 0 , 0 , 0 , 0 , 0 , 0 , 0, 0],
]) # this is a random matrix that I made up, it doesnt mean anything

In [3]:
k = v = q # q, k, v = x/input

In [4]:
k_t = k.T # K Transpose

Confirming the shapes of our matricies

In [5]:
q.shape

(5, 10)

In [6]:
k_t.shape

(10, 5)

In [7]:
v.shape

(5, 10)

Now lets do the first step which is 
$QK^{T}$

In [8]:
qk_t = q @ k_t

Lets check the shape of this and the actual values.

In [9]:
qk_t.shape

(5, 5)

In [10]:
qk_t

matrix([[21317, 19396, 23246,     0,     0],
        [19396, 39657, 24746,     0,     0],
        [23246, 24746, 37044,     0,     0],
        [    0,     0,     0,     0,     0],
        [    0,     0,     0,     0,     0]])

Now we are going to do this part $\frac{QK^{T}}{\sqrt{d_k}}$ in the equation.

In [11]:
seq_len, d_model = q.shape

In [12]:
heads = 5

In [13]:
d_k = d_model/heads

In [14]:
qk_t_dk = qk_t / (d_model ** 2)

In [15]:
qk_t_dk.shape

(5, 5)

In [16]:
qk_t_dk

matrix([[213.17, 193.96, 232.46,   0.  ,   0.  ],
        [193.96, 396.57, 247.46,   0.  ,   0.  ],
        [232.46, 247.46, 370.44,   0.  ,   0.  ],
        [  0.  ,   0.  ,   0.  ,   0.  ,   0.  ],
        [  0.  ,   0.  ,   0.  ,   0.  ,   0.  ]])

Now we are going to apply softmax to $\frac{QK^{T}}{\sqrt{d_k}}$ which will give us $softmax (\frac{QK^{T}}{\sqrt{d_k}})$

Alright we have applied softmax now all that is left to finish computing attention is multplying $softmax (\frac{QK^{T}}{\sqrt{d_k}})$ by $V$. 

We have now computed our attention scores! 

Now that was regular self attention which is where we process the matrix all at once. Now lets get into multi head attention which is what transformers use.

#### Multi Head Attention
Multi head attention (MHA) is very very similar but instead of calculating attention all at once we split the matrix in `h` heads. We defined a variable of $\text{h heads} = 5$ above. 

The idea with MHA is that we calcualte attention for each head. This way each head contains a small part of each query key and value and each head can .... The once we have the attention scores computed for each of the heads we combine them back to get the attention matrix. 

So we still have our matrix from before which is. 

In [17]:
q

matrix([[ 8, 63, 39, 36, 81,  3, 73,  8, 50,  2],
        [61, 90, 99, 92, 42, 25, 30,  5,  4, 79],
        [11, 99, 10, 34, 72,  9, 74, 84, 65, 62],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0]])

Now we are going to split uour matrix q into h heads. 

In [18]:
q_heads = np.asarray(q).reshape(seq_len, heads, int(d_k)).transpose(1, 0, 2)

This gives us our Q matrix split into heads as 

In [19]:
q_heads.shape

(5, 5, 2)

In [20]:
q_heads

array([[[ 8, 63],
        [61, 90],
        [11, 99],
        [ 0,  0],
        [ 0,  0]],

       [[39, 36],
        [99, 92],
        [10, 34],
        [ 0,  0],
        [ 0,  0]],

       [[81,  3],
        [42, 25],
        [72,  9],
        [ 0,  0],
        [ 0,  0]],

       [[73,  8],
        [30,  5],
        [74, 84],
        [ 0,  0],
        [ 0,  0]],

       [[50,  2],
        [ 4, 79],
        [65, 62],
        [ 0,  0],
        [ 0,  0]]])

We do this for $K^T$ and $V$ and basically do the same as we did above but for every head. I will not do it here tho but you can view the code for this in `./transformer/MultiHeadAttention.py`. Then in the end we concat and multiply by a matrix $W^O$. As mentioned the reason for this is because now each head sees the full sequence but a small part of its embedding, meaning each head can get unique information from the sequence. 

In the end this is our equation. 

$$ \forall i \in \text{h heads} \hspace{10px} head_i =  AttentionHead(Q_i,K_i, V_i) = softmax(\frac{Q_iK^{T}_i}{\sqrt{d_k}})V_i $$

$$ Attention(Q,K,V) = concat_{n=1}^{\text{h heads}}(head_i)W^{O}$$

### Projection Layer

### Recap
To fully understand what weve done lets go through the shape change for a input. We are going to assume only 1 sample for simplicity. 

#### Encoder Inputs
We get our inputs for our encoder which gives us shape $(seq \_ len, d_{model})$

#### Positional Encoding
We then get information on the positions we keep our shape of $(seq \_ len, d_{model})$

#### Encoder Blocks
- We apply layer norm which is so we keep our $(seq \_ len, d_{model})$
- After that we pass to our attention block. The input x gets duplicated into q,k,v.
- Each q,k,v is size $(seq \_ len, d_{model})$ and goes through a linear layer of shape $(d_{model}, d_{model})$ so our shape stays at $(seq \_ len, d_{model})$
- We then reshape this from $(seq \_ len, d_{model})$ into $(n\_heads, seq\_len, d_k)$ where $d_k = \frac{d_{model}}{n_{heads}}$. This is so we can perform attention on each head.

## Training 
So the goal is to train a model that learns Spansih to English and Nahuatl. To do this I got some insipration from this. 
Because Nahuatl is a low resource language ( samples) the plan is to train the model on the Spanish to English dataset which is very large. This way in a second phase of training the encoder has already learned patterns of spanish and can focus on learning the patterns of Nahuatl in the decoder. 

### Phase 1
So for phase 1 we have our Spanish to English dataset which is at about n samples. Below are the hyper parameters I choose for the model.

```
BATCH_SIZE=32,
EPOCHS=50,
LR=3e-4,
SEQ_LEN=128,
D_MODEL=512,
D_FF=2048,
H=8,
N=6,
DROPOUT_SCHEDULE={0: 0.15, 15: 0.25, 30: 0.3},
WEIGHT_DECAY=0.05

```

Initially the model was overfitting by a lot so I chose a dropout rate schedule. The idea is that the first few iterations have minimal dropout so the model can learn as much as possible quickly. At around iteration 15 it starts to overfit so we introduce a higher level of dropout so that the model can truly learn. To further prevent overfitting at around iteration 30 we update the dropout to be higher. This way we ensure that the model learns and does not overfit in the final iterations. The epochs was set about 100 but I really only ran it 50. I continued the training and got a better eval loss at around 70 but it really is not much improvement. This took like about 12+ hours if i remember correctly. 

## Phase 2
For phase two we had a lot less data. The dataset is of size n but on top of this I added n Spanish to English samples. So our phase 2 training included Spanish to English and Spanish to Nahuatl samples. 

When training we used the best checkpoint from phase 1 which was `70`. 

## Results 

## Conclusion